In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from math import radians, cos, sin, sqrt, atan2
from matplotlib.patches import FancyArrowPatch
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px



# -- Raw to CSV --

In [2]:
import os
import numpy as np
import pandas as pd
from pioneer.das.api.platform import Platform
from math import radians, sin, cos, sqrt, atan2
import uuid
from tqdm.notebook import tqdm  # Pour Jupyter

# === Distance haversine ===
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = radians(lat1), radians(lat2)
    d_phi = radians(lat2 - lat1)
    d_lambda = radians(lon2 - lon1)
    a = sin(d_phi / 2) ** 2 + cos(phi1) * cos(phi2) * sin(d_lambda / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# === Générer un ID piéton unique ===
def generate_id():
    return str(uuid.uuid4())[:8]

# === Mappage météo ===
def get_weather_class(folder_name, weather_csv_path):
    try:
        df_weather = pd.read_csv(weather_csv_path)
        row = df_weather[df_weather['folder'] == folder_name]
        if row.empty:
            return None
        row = row.iloc[0]
        if not pd.isna(row.get('snow')) or not pd.isna(row.get('twilight')):
            return None
        if not pd.isna(row.get('night')):
            return 'night'
        elif not pd.isna(row.get('rain')) or not pd.isna(row.get('clouds')):
            return 'rain'
        elif not pd.isna(row.get('day')):
            return 'clear'
        else:
            return None
    except Exception as e:
        print(f"Erreur météo pour {folder_name}: {e}")
        return None

# === Dossiers ===
root_dataset = r'C:\Users\svictor\Documents\PIXSET\dataset'
output_csv_dir = r'C:\Users\svictor\Documents\PIXSET\output'
weather_csv_path = os.path.join(root_dataset, 'weather.csv')
os.makedirs(output_csv_dir, exist_ok=True)

# === Traitement des dossiers ===
folder_list = [f for f in os.listdir(root_dataset) if os.path.isdir(os.path.join(root_dataset, f)) and f != "predictions"]

for folder_name in tqdm(folder_list, desc="📂 Dossiers"):
    dataset_path = os.path.join(root_dataset, folder_name)

    try:
        pf = Platform(dataset_path)
        sync = pf.synchronized(sync_labels=[
            'flir_bbfc_flimg',
            'pixell_bfc_box3d-deepen',
            'sbgekinox_bcc_navposvel'
        ], tolerance_us=1e6)
    except Exception as e:
        print(f"⚠️ Erreur chargement {folder_name}: {e}")
        continue

    tracked_pedestrians = {}
    ASSOCIATION_THRESHOLD = 1.5  # mètres
    annotations = []

    weather_class = get_weather_class(folder_name, weather_csv_path)
    if weather_class is None:
        print(f"⛔ Dossier ignoré (météo): {folder_name}")
        continue

    for frame_idx in tqdm(range(len(sync)), leave=False, desc=f"🎞️ {folder_name}"):
        frame = sync[frame_idx]

        try:
            ego_sample = frame['sbgekinox_bcc_navposvel']
            raw_data = ego_sample.raw
            ego_lat = raw_data['latitude']
            ego_lon = raw_data['longitude']
            ego_alt = raw_data['altitude']

            v_n, v_e, v_d = raw_data['velocity_n'], raw_data['velocity_e'], raw_data['velocity_d']
            ego_speed_mps = sqrt(v_n**2 + v_e**2 + v_d**2)

            boxes3d = frame['pixell_bfc_box3d-deepen']
            categories = boxes3d.get_categories()
            dimensions = boxes3d.get_dimensions()
            centers = boxes3d.get_centers()

            for i, category in enumerate(categories):
                if category != 'pedestrian':
                    continue

                center = centers[i]
                transform = boxes3d.compute_transform(i)
                world_pos = boxes3d.transform_pts(transform, np.array([center]))[0]

                d_north = world_pos[0]
                d_east = world_pos[1]
                d_lat = d_north / 111320
                d_lon = d_east / (40075000 * cos(radians(ego_lat)) / 360)
                ped_lat = ego_lat + d_lat
                ped_lon = ego_lon + d_lon

                dist_m = haversine_distance(ego_lat, ego_lon, ped_lat, ped_lon)

                matched_id = None
                for pid, pdata in tracked_pedestrians.items():
                    prev = pdata["position"]
                    if np.linalg.norm(np.array(prev) - np.array(world_pos[:2])) < ASSOCIATION_THRESHOLD:
                        matched_id = pid
                        break

                if matched_id is None:
                    matched_id = generate_id()

                tracked_pedestrians[matched_id] = {
                    "position": world_pos[:2],
                    "last_position": world_pos[:2],
                    "frame": frame_idx
                }

                annotations.append({
                    "frame_id": frame_idx,
                    "taille_cm": dimensions[i][2] * 100,
                    "dist_m": dist_m,
                    "ego_speed_mps": ego_speed_mps,
                    "ego_lat": ego_lat,
                    "ego_lon": ego_lon,
                    "ped_lat": ped_lat,
                    "ped_lon": ped_lon,
                    "weather": weather_class,
                    "pedestrian_id": matched_id
                })

        except Exception as e:
            print(f"⚠️ Erreur frame {frame_idx} dans {folder_name}: {e}")
            continue

    df = pd.DataFrame(annotations)
    csv_path = os.path.join(output_csv_dir, f"{folder_name}.csv")
    df.to_csv(csv_path, index=False)


📂 Dossiers:   0%|          | 0/96 [00:00<?, ?it/s]


Loading sensors: 100%|███████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 12.15it/s]

Synchronizing: 100%|███████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 5971.96it/s]


🎞️ 20200610_185206_part1_5095_5195:   0%|          | 0/102 [00:00<?, ?it/s]


Loading sensors: 100%|███████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 12.25it/s]

Synchronizing: 100%|███████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 1940.61it/s]


🎞️ 20200610_185206_part1_9850_10050:   0%|          | 0/203 [00:00<?, ?it/s]

...

⛔ Dossier ignoré (météo): 20200805_000536_part47_2225_2325



Loading sensors: 100%|███████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 11.54it/s]

Synchronizing: 100%|█████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<?, ?it/s]


⛔ Dossier ignoré (météo): 20200805_000536_part47_5292_5622



Loading sensors: 100%|███████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 11.59it/s]

Synchronizing: 100%|█████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<?, ?it/s]

⛔ Dossier ignoré (météo): 20200805_002607_part48_2083_2282


# -- Annotation Crossing --

In [3]:
import os
import pandas as pd
from math import radians, cos, sin, sqrt, atan2
from tqdm.notebook import tqdm

# Dossier contenant les CSV
csv_dir = r"C:\Users\svictor\Documents\PIXSET\output"

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # rayon Terre en mètres
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlambda = radians(lon2 - lon1)
    a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dlambda/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

def ccw(A, B, C):
    return (C[1]-A[1])*(B[0]-A[0]) > (B[1]-A[1])*(C[0]-A[0])

def segments_intersect(p1, p2, q1, q2):
    return (ccw(p1, q1, q2) != ccw(p2, q1, q2)) and (ccw(p1, p2, q1) != ccw(p1, p2, q2))

def compute_side(ped, ego1, ego2):
    vx = ped['ped_lon'] - ego1['ego_lon']
    vy = ped['ped_lat'] - ego1['ego_lat']
    dx = ego2['ego_lon'] - ego1['ego_lon']
    dy = ego2['ego_lat'] - ego1['ego_lat']
    cross = dx * vy - dy * vx
    return 'opposite' if cross < 0 else 'same'

all_files = [f for f in os.listdir(csv_dir) if f.endswith(".csv")]

for file in tqdm(all_files, desc="Fichiers CSV"):
    path = os.path.join(csv_dir, file)
    try:
        df = pd.read_csv(path)
    except Exception:
        continue

    if df.empty or 'pedestrian_id' not in df.columns:
        continue

    df = df.sort_values('frame_id')
    df['crossing'] = False
    df['crossing_frame_id'] = pd.NA

    ped_ids = df['pedestrian_id'].dropna().unique()
    for ped_id in tqdm(ped_ids, desc=f"Piétons dans {file}", leave=False):
        df_ped = df[df['pedestrian_id'] == ped_id].sort_values('frame_id')
        df_veh = df[df['frame_id'].isin(df_ped['frame_id'])].sort_values('frame_id')

        found = False
        cross_frame = None

        # Progression frame par frame sur ce piéton
        for i in tqdm(range(len(df_ped) - 1), desc=f"Frames piéton {ped_id}", leave=False):
            p1 = (df_ped.iloc[i]['ped_lon'], df_ped.iloc[i]['ped_lat'])
            p2 = (df_ped.iloc[i+1]['ped_lon'], df_ped.iloc[i+1]['ped_lat'])

            for j in range(len(df_veh) - 1):
                q1 = (df_veh.iloc[j]['ego_lon'], df_veh.iloc[j]['ego_lat'])
                q2 = (df_veh.iloc[j+1]['ego_lon'], df_veh.iloc[j+1]['ego_lat'])

                if segments_intersect(p1, p2, q1, q2):
                    cross_frame = df_ped.iloc[i]['frame_id']
                    found = True
                    break
            if found:
                break

        if not found:
            continue

        try:
            pre = df_ped[df_ped['frame_id'] < cross_frame].iloc[-1]
            ego1 = df_veh[df_veh['frame_id'] < cross_frame].iloc[-2]
            ego2 = df_veh[df_veh['frame_id'] < cross_frame].iloc[-1]
        except IndexError:
            continue

        side = compute_side(pre, ego1, ego2)
        dist_required = 7 if side == 'opposite' else 3.5

        df_ped = df_ped.copy()
        cum_dist = [0]
        for i in range(1, len(df_ped)):
            d = haversine(
                df_ped.iloc[i-1]['ped_lat'], df_ped.iloc[i-1]['ped_lon'],
                df_ped.iloc[i]['ped_lat'], df_ped.iloc[i]['ped_lon']
            )
            cum_dist.append(cum_dist[-1] + d)
        df_ped['cum_dist'] = cum_dist

        cross_pos = df_ped[df_ped['frame_id'] == cross_frame].index[0]
        cross_idx = df_ped.index.get_loc(cross_pos)

        start_idx = cross_idx
        while start_idx > 0 and (cum_dist[cross_idx] - cum_dist[start_idx]) < dist_required:
            start_idx -= 1

        end_idx = cross_idx
        while end_idx < len(df_ped)-1 and (cum_dist[end_idx] - cum_dist[cross_idx]) < dist_required:
            end_idx += 1

        crossing_frames = df_ped.iloc[start_idx:end_idx + 1]['frame_id']
        df.loc[(df['pedestrian_id'] == ped_id) & (df['frame_id'].isin(crossing_frames)), 'crossing'] = True
        df.loc[(df['pedestrian_id'] == ped_id) & (df['frame_id'].isin(crossing_frames)), 'crossing_frame_id'] = cross_frame

    df['crossing_frame_id'] = df['crossing_frame_id'].astype('Int64')
    df.to_csv(path, index=False)


Fichiers CSV:   0%|          | 0/88 [00:00<?, ?it/s]

Piétons dans 20200610_185206_part1_5095_5195.csv:   0%|          | 0/49 [00:00<?, ?it/s]

Frames piéton 996a7449:   0%|          | 0/47 [00:00<?, ?it/s]

Frames piéton b296ad1d:   0%|          | 0/84 [00:00<?, ?it/s]

Frames piéton fbbc8ada:   0%|          | 0/46 [00:00<?, ?it/s]

...

Frames piéton 44764735:   0%|          | 0/23 [00:00<?, ?it/s]

Frames piéton c24fd883:   0%|          | 0/2 [00:00<?, ?it/s]

Frames piéton 5079b0a2:   0%|          | 0/244 [00:00<?, ?it/s]

Frames piéton 4ad0471e:   0%|          | 0/558 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Dossier racine contenant les sous-dossiers avec les CSV
root_dir = r"C:\Users\svictor\Documents\PIXSET\output"

for subdir, _, files in os.walk(root_dir):
    for file in files:
        if not file.endswith(".csv"):
            continue

        csv_path = os.path.join(subdir, file)
        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"Erreur lecture {file}: {e}")
            continue

        if df.empty or 'pedestrian_id' not in df.columns:
            continue

        # Trier par frame
        df = df.sort_values('frame_id')

        fig, ax = plt.subplots(figsize=(10, 8))

        # Trajectoire du véhicule
        ax.plot(df['ego_lon'], df['ego_lat'], color='blue', label='Véhicule')

        # Pour éviter doublons dans la légende des piétons crossing
        crossing_labels_done = set()

        # Trajectoires piétons
        for ped_id in df['pedestrian_id'].dropna().unique():
            df_ped = df[df['pedestrian_id'] == ped_id]
            if df_ped.empty:
                continue

            # Trajectoire crossing == True
            df_cross = df_ped[df_ped['crossing'] == True]
            df_noncross = df_ped[df_ped['crossing'] == False]

            if not df_noncross.empty:
                ax.plot(df_noncross['ped_lon'], df_noncross['ped_lat'],
                        color='lightgray', linestyle='dashed', linewidth=1)

            if not df_cross.empty:
                label = None
                if ped_id not in crossing_labels_done:
                    label = f'Piéton {ped_id} (crossing)'
                    crossing_labels_done.add(ped_id)
                ax.plot(df_cross['ped_lon'], df_cross['ped_lat'],
                        color='red', linewidth=2, label=label)

        ax.set_title(f"Trajectoires - {file}")
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.axis('equal')
        ax.legend()
        ax.grid(True)

        plt.show()


# -- Crossing Decision --

In [ ]:
import os
import numpy as np
import pandas as pd
from pioneer.das.api.platform import Platform
from math import radians, sin, cos, sqrt, atan2
import uuid
from tqdm.notebook import tqdm  # Pour Jupyter

# === Distance haversine ===
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = radians(lat1), radians(lat2)
    d_phi = radians(lat2 - lat1)
    d_lambda = radians(lon2 - lon1)
    a = sin(d_phi / 2) ** 2 + cos(phi1) * cos(phi2) * sin(d_lambda / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# === Générer un ID piéton unique ===
def generate_id():
    return str(uuid.uuid4())[:8]

# === Mappage météo ===
def get_weather_class(folder_name, weather_csv_path):
    try:
        df_weather = pd.read_csv(weather_csv_path)
        row = df_weather[df_weather['folder'] == folder_name]
        if row.empty:
            return None
        row = row.iloc[0]
        if not pd.isna(row.get('snow')) or not pd.isna(row.get('twilight')):
            return None
        if not pd.isna(row.get('night')):
            return 'night'
        elif not pd.isna(row.get('rain')) or not pd.isna(row.get('clouds')):
            return 'rain'
        elif not pd.isna(row.get('day')):
            return 'clear'
        else:
            return None
    except Exception as e:
        print(f"Erreur météo pour {folder_name}: {e}")
        return None

# === Dossiers ===
root_dataset = r'C:\Users\svictor\Documents\PIXSET\dataset'
output_csv_dir = r'C:\Users\svictor\Documents\PIXSET\output'
weather_csv_path = os.path.join(root_dataset, 'weather.csv')
os.makedirs(output_csv_dir, exist_ok=True)

# === Traitement des dossiers ===
folder_list = [f for f in os.listdir(root_dataset) if os.path.isdir(os.path.join(root_dataset, f)) and f != "predictions"]

for folder_name in tqdm(folder_list, desc="📂 Dossiers"):
    dataset_path = os.path.join(root_dataset, folder_name)

    try:
        pf = Platform(dataset_path)
        sync = pf.synchronized(sync_labels=[
            'flir_bbfc_flimg',
            'pixell_bfc_box3d-deepen',
            'sbgekinox_bcc_navposvel'
        ], tolerance_us=1e6)
    except Exception as e:
        print(f"⚠️ Erreur chargement {folder_name}: {e}")
        continue

    tracked_pedestrians = {}
    ASSOCIATION_THRESHOLD = 1.5  # mètres
    annotations = []

    weather_class = get_weather_class(folder_name, weather_csv_path)
    if weather_class is None:
        print(f"⛔ Dossier ignoré (météo): {folder_name}")
        continue

    for frame_idx in tqdm(range(len(sync)), leave=False, desc=f"🎞️ {folder_name}"):
        frame = sync[frame_idx]

        try:
            ego_sample = frame['sbgekinox_bcc_navposvel']
            raw_data = ego_sample.raw
            ego_lat = raw_data['latitude']
            ego_lon = raw_data['longitude']
            ego_alt = raw_data['altitude']

            v_n, v_e, v_d = raw_data['velocity_n'], raw_data['velocity_e'], raw_data['velocity_d']
            ego_speed_mps = sqrt(v_n**2 + v_e**2 + v_d**2)

            boxes3d = frame['pixell_bfc_box3d-deepen']
            categories = boxes3d.get_categories()
            dimensions = boxes3d.get_dimensions()
            centers = boxes3d.get_centers()

            for i, category in enumerate(categories):
                if category != 'pedestrian':
                    continue

                center = centers[i]
                transform = boxes3d.compute_transform(i)
                world_pos = boxes3d.transform_pts(transform, np.array([center]))[0]

                d_north = world_pos[0]
                d_east = world_pos[1]
                d_lat = d_north / 111320
                d_lon = d_east / (40075000 * cos(radians(ego_lat)) / 360)
                ped_lat = ego_lat + d_lat
                ped_lon = ego_lon + d_lon

                dist_m = haversine_distance(ego_lat, ego_lon, ped_lat, ped_lon)

                matched_id = None
                for pid, pdata in tracked_pedestrians.items():
                    prev = pdata["position"]
                    if np.linalg.norm(np.array(prev) - np.array(world_pos[:2])) < ASSOCIATION_THRESHOLD:
                        matched_id = pid
                        break

                if matched_id is None:
                    matched_id = generate_id()

                tracked_pedestrians[matched_id] = {
                    "position": world_pos[:2],
                    "last_position": world_pos[:2],
                    "frame": frame_idx
                }

                annotations.append({
                    "frame_id": frame_idx,
                    "taille_cm": dimensions[i][2] * 100,
                    "dist_m": dist_m,
                    "ego_speed_mps": ego_speed_mps,
                    "ego_lat": ego_lat,
                    "ego_lon": ego_lon,
                    "ped_lat": ped_lat,
                    "ped_lon": ped_lon,
                    "weather": weather_class,
                    "pedestrian_id": matched_id
                })

        except Exception as e:
            print(f"⚠️ Erreur frame {frame_idx} dans {folder_name}: {e}")
            continue

    df = pd.DataFrame(annotations)
    csv_path = os.path.join(output_csv_dir, f"{folder_name}.csv")
    df.to_csv(csv_path, index=False)
    print(f"💾 CSV sauvegardé : {csv_path}")



In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Dossier racine contenant les sous-dossiers avec les CSV
root_dir = r"C:\Users\svictor\Documents\PIXSET\output"

for subdir, _, files in os.walk(root_dir):
    for file in files:
        if not file.endswith(".csv"):
            continue

        csv_path = os.path.join(subdir, file)
        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"Erreur lecture {file}: {e}")
            continue

        if df.empty or 'pedestrian_id' not in df.columns:
            continue

        # Vérifier que les colonnes nécessaires existent
        required_cols = ['frame_id', 'ego_lon', 'ego_lat', 'pedestrian_id',
                         'ped_lon', 'ped_lat', 'crossing', 'prediction']
        if not all(col in df.columns for col in required_cols):
            print(f"Colonnes manquantes dans {file}, ignoré.")
            continue

        # Trier par frame
        df = df.sort_values('frame_id')

        fig, ax = plt.subplots(figsize=(10, 8))

        # Trajectoire du véhicule
        ax.plot(df['ego_lon'], df['ego_lat'], color='blue', label='Véhicule')

        # Pour éviter doublons dans la légende
        crossing_labels_done = set()

        # Trajectoires piétons
        for ped_id in df['pedestrian_id'].dropna().unique():
            df_ped = df[df['pedestrian_id'] == ped_id]
            if df_ped.empty:
                continue

            # Vérité terrain
            df_cross = df_ped[df_ped['crossing'] == True]
            df_noncross = df_ped[df_ped['crossing'] == False]

            # Prédiction crossing
            df_pred_cross = df_ped[df_ped['prediction'] == True]

            # Non-crossing (vérité terrain)
            if not df_noncross.empty:
                ax.plot(df_noncross['ped_lon'], df_noncross['ped_lat'],
                        color='lightgray', linestyle='dashed', linewidth=1)

            # Crossing (vérité terrain)
            if not df_cross.empty:
                label = None
                if ped_id not in crossing_labels_done:
                    label = f'Piéton {ped_id} (crossing GT)'
                    crossing_labels_done.add(ped_id)
                ax.plot(df_cross['ped_lon'], df_cross['ped_lat'],
                        color='red', linewidth=2, label=label)

            # Prédiction crossing
            if not df_pred_cross.empty:
                ax.plot(df_pred_cross['ped_lon'], df_pred_cross['ped_lat'],
                        color='green', linewidth=2, linestyle='solid', alpha=0.6,
                        label=f'Prediction {ped_id}')

        ax.set_title(f"Trajectoires - {file}")
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.axis('equal')
        ax.legend()
        ax.grid(True)

        plt.show()
